# exp_4b — Experiment 4B: Ensemble Modeling with CNN, MobileNetV2 and VGG16

Source-based notebook converted from the uploaded experiment code document.


In [ ]:
import zipfile
import os
import shutil

In [ ]:
zip_file = r"D:\DL DATASET\dogs_vs_cats_subset (1) (1).zip"

extract_path = r"D:\DL DATASET\dogs_vs_cats_subset"

In [ ]:
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, VGG16
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.models import Sequential, Model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

IMG_SIZE=(224,224)
BATCH_SIZE=32
EPOCHS=5

In [ ]:
train_dir = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\train"

val_dir = r"D:\DL DATASET\dogs_vs_cats_subset\dogs_vs_cats_subset\val"

In [ ]:
train_gen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True)

val_gen=ImageDataGenerator(rescale=1./255)

train_ds=train_gen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary")

val_ds=val_gen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False)

In [ ]:
cnn=Sequential([
    tf.keras.Input(shape=(224,224,3)),
    Conv2D(32,3,activation="relu"),
    MaxPooling2D(),
    Conv2D(64,3,activation="relu"),
    MaxPooling2D(),
    Conv2D(128,3,activation="relu"),
    MaxPooling2D(),
    Flatten(),
    Dense(128,activation="relu"),
    Dropout(0.5),
    Dense(1,activation="sigmoid")
])

cnn.compile(optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"])

cnn_history=cnn.fit(train_ds,
                    validation_data=val_ds,
                    epochs=EPOCHS)

In [ ]:
base=MobileNetV2(weights="imagenet",
                  include_top=False,
                  input_shape=(224,224,3))

base.trainable=False

x=GlobalAveragePooling2D()(base.output)
x=Dense(128,activation="relu")(x)
x=Dropout(0.5)(x)
output=Dense(1,activation="sigmoid")(x)

mobilenet=Model(base.input,output)

mobilenet.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])

mobilenet.fit(train_ds,
              validation_data=val_ds,
              epochs=EPOCHS)

# Fine-tuning
base.trainable=True
for layer in base.layers[:-30]:
    layer.trainable=False

mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"])

mobilenet.fit(train_ds,
              validation_data=val_ds,
              epochs=2)

In [ ]:
base2=VGG16(weights="imagenet",
             include_top=False,
             input_shape=(224,224,3))

base2.trainable=False

y = GlobalAveragePooling2D()(base2.output)
y=Dense(128,activation="relu")(y)
y=Dropout(0.5)(y)
output2=Dense(1,activation="sigmoid")(y)

vgg=Model(base2.input,output2)

vgg.compile(optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"])

vgg.fit(train_ds,
        validation_data=val_ds,
        epochs=EPOCHS)

In [ ]:
for name,model in [("CNN",cnn),
                   ("MobileNetV2",mobilenet),
                   ("VGG16",vgg)]:
    loss,acc=model.evaluate(val_ds,verbose=0)
    print(f"{name}: Accuracy = {acc:.4f}")

In [ ]:
val_ds.reset()
p1=cnn.predict(val_ds,verbose=0)

val_ds.reset()
p2=mobilenet.predict(val_ds,verbose=0)

val_ds.reset()
p3=vgg.predict(val_ds,verbose=0)

ensemble=((p1+p2+p3)/3>0.5).astype(int).flatten()

true=val_ds.classes

print("Ensemble Accuracy:",
      accuracy_score(true,ensemble))

print(classification_report(
    true,
    ensemble,
    target_names=["Cat","Dog"]))

In [ ]:
cm=confusion_matrix(true,ensemble)

plt.figure(figsize=(5,4))
plt.imshow(cm,cmap="Blues")
plt.colorbar()
plt.xticks([0,1],["Cat","Dog"])
plt.yticks([0,1],["Cat","Dog"])

for i in range(2):
    for j in range(2):
        plt.text(j,i,str(cm[i,j]),
                 ha="center",
                 va="center")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Ensemble Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

val_ds.reset()
cnn_pred = (cnn.predict(val_ds, verbose=0) > 0.5).astype(int).flatten()

val_ds.reset()
mobile_pred = (mobilenet.predict(val_ds, verbose=0) > 0.5).astype(int).flatten()

val_ds.reset()
vgg_pred = (vgg.predict(val_ds, verbose=0) > 0.5).astype(int).flatten()

print("CNN:", accuracy_score(true, cnn_pred))
print("MobileNetV2:", accuracy_score(true, mobile_pred))
print("VGG16:", accuracy_score(true, vgg_pred))
print("Ensemble:", accuracy_score(true, ensemble))

In [ ]:
import numpy as np

cnn_pred = (p1 > 0.5).astype(int).flatten()
mobile_pred = (p2 > 0.5).astype(int).flatten()
vgg_pred = (p3 > 0.5).astype(int).flatten()

print("CNN mistakes:", np.sum(cnn_pred != true))
print("MobileNet mistakes:", np.sum(mobile_pred != true))
print("VGG mistakes:", np.sum(vgg_pred != true))
print("Ensemble mistakes:", np.sum(ensemble != true))

In [ ]:
mobile_wrong = np.where(mobile_pred != true)[0]
print(mobile_wrong)

In [ ]:
for idx in mobile_wrong:
    print(f"Image {idx}")
    print("True Label:", true[idx])
    print("CNN:", cnn_pred[idx])
    print("MobileNet:", mobile_pred[idx])
    print("VGG:", vgg_pred[idx])
    print("Ensemble:", ensemble[idx])
    print("-" * 30)

In [ ]:
print("Image 31")

print("CNN Probability       :", p1[31][0])
print("MobileNet Probability :", p2[31][0])
print("VGG16 Probability     :", p3[31][0])

print("Average :", (p1[31][0]+p2[31][0]+p3[31][0])/3)